In [1]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime, timedelta

In [2]:
RAW_DIR = "raw_data"
os.makedirs(RAW_DIR, exist_ok=True)

In [3]:
def get_last_date(ticker):
    """Get the last saved date from existing CSV, or None if not found."""
    path = f"{RAW_DIR}/{ticker}.csv"
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, parse_dates=["Date"])
    return df["Date"].max().strftime("%Y-%m-%d") if not df.empty else None

In [4]:
def save_raw(ticker, new_df):
    """Append new data to existing CSV or create a new one."""
    path = f"{RAW_DIR}/{ticker}.csv"
    new_df = new_df.reset_index()
    new_df.columns = new_df.columns.get_level_values(0)

    if os.path.exists(path):
        existing = pd.read_csv(path, parse_dates=["Date"])
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset="Date", keep="last")
        combined = combined.sort_values("Date")
        combined.to_csv(path, index=False)
    else:
        new_df.to_csv(path, index=False)

In [5]:
def save_raw(ticker, new_df):
    """Append new data to existing CSV or create a new one."""
    path = f"{RAW_DIR}/{ticker}.csv"
    new_df = new_df.reset_index()
    new_df.columns = new_df.columns.get_level_values(0)

    if os.path.exists(path):
        existing = pd.read_csv(path, parse_dates=["Date"])
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset="Date", keep="last")
        combined = combined.sort_values("Date")
        combined.to_csv(path, index=False)
    else:
        new_df.to_csv(path, index=False)


def extract(ticker, start="2020-01-01"):
    """
    Extract daily stock data with incremental updates.
    - First run  → fetch from start date
    - Next runs  → fetch only new data since last saved date
    """
    ticker  = ticker.upper()
    last    = get_last_date(ticker)
    today   = datetime.today().strftime("%Y-%m-%d")

    if last:
        fetch_from = (pd.to_datetime(last) + timedelta(days=1)).strftime("%Y-%m-%d")
        if fetch_from >= today:
            print(f"[{ticker}] Already up to date (last: {last})")
            return
    else:
        fetch_from = start

    print(f"[{ticker}] Fetching {fetch_from} → {today}")

    raw = yf.download(ticker, start=fetch_from, interval="1d",
                      auto_adjust=True, progress=False)

    if raw.empty:
        print(f"[{ticker}] No new data found.")
        return

    save_raw(ticker, raw)
    print(f"[{ticker}] Saved {len(raw)} rows → raw_data/{ticker}.csv")

```
AAPL: Apple Inc.
MSFT: Microsoft Corporation
GOOGL: Alphabet Inc.
NVDA: NVIDIA Corporation
TSLA: Tesla, Inc.
META: Meta Platforms, Inc.
NFLX: Netflix, Inc.
AMZN: Amazon.com, Inc.
AMD: Advanced Micro Devices, Inc.
TEAM: Atlassian Corporation
```

In [6]:
for ticker in ["AAPL", "MSFT", "GOOGL","META","TSLA","NVDA","NFLX","AMZN","AMD","TEAM"]:
    extract(ticker)

[AAPL] Already up to date (last: 2026-05-06)
[MSFT] Already up to date (last: 2026-05-06)
[GOOGL] Already up to date (last: 2026-05-06)
[META] Already up to date (last: 2026-05-06)
[TSLA] Already up to date (last: 2026-05-06)
[NVDA] Already up to date (last: 2026-05-06)
[NFLX] Already up to date (last: 2026-05-06)
[AMZN] Already up to date (last: 2026-05-06)
[AMD] Already up to date (last: 2026-05-06)
[TEAM] Already up to date (last: 2026-05-06)
